# Part II - Implement the A* Search Algorithm for City Navigation

**Module:** AAC6153/ARA6113 AI Fundamentals / Introduction to AI  
**Assignment weight:** 10% for Part II

## What this notebook delivers

This notebook:
1. represents the supplied city map as a **Python dictionary**;
2. implements a complete **A* Search** algorithm;
3. outputs the **shortest path, total cost, and cities along the path**;
4. uses a **provably admissible and consistent heuristic**;
5. independently verifies the result with **Dijkstra's algorithm** so that the reported route is not accidentally non-optimal.

> **Important ambiguity in the assignment PDF:** the written task says "between cities A and B", but the supplied map does not label any cities A or B. The code is therefore written to accept **any two named cities**. The demonstration below uses **Seattle -> Miami**; if the lecturer later specifies different endpoints, change only `START_CITY` and `GOAL_CITY`.

## 1. What A* Search means

A* evaluates each candidate city with:

**f(n) = g(n) + h(n)**

- `g(n)`: the actual cost from the start city to city `n`.
- `h(n)`: an estimate of the remaining cost from `n` to the goal.
- `f(n)`: estimated total cost of a route through `n`.

The key correctness rule is that `h(n)` must not **overestimate** the true remaining cost. Such a heuristic is called **admissible**. A consistent heuristic gives the additional guarantee that the first time a node is removed from the priority queue with its best cost, its route cost is final.

### Why we do NOT invent geographic straight-line distances

The assignment supplies edge costs but does not supply city coordinates or a heuristic table. Using guessed geographic coordinates would introduce unsupported assumptions. Instead, this project uses a graph-only heuristic that is mathematically guaranteed to be safe:

**h(n) = minimum-edge-cost x minimum-number-of-edges from n to the goal**

The smallest edge on the supplied map costs 50. Because every edge costs at least 50, a route containing `k` edges must cost at least `50*k`. Therefore this heuristic can never overestimate the true remaining cost.

## 2. Map representation

The map is an **undirected weighted graph**: an edge can be travelled in either direction and its number is the edge cost.

The dictionary below stores every connection in both directions. This is important: storing only one direction would silently change the problem represented by the lecturer's map.

In [1]:
# The supplied map represented as an undirected weighted adjacency dictionary.
graph = {
    "Atlanta": {
        "Dallas": 721,
        "Houston": 702,
        "Chicago": 588,
        "Washington": 543,
        "Miami": 604,
    },
    "Boston": {
        "Detroit": 613,
        "New York": 190,
    },
    "Chicago": {
        "Seattle": 1737,
        "Riverside": 1704,
        "Dallas": 805,
        "Atlanta": 588,
        "Detroit": 238,
    },
    "Dallas": {
        "Phoenix": 887,
        "Chicago": 805,
        "Houston": 225,
        "Atlanta": 721,
    },
    "Detroit": {
        "Chicago": 238,
        "Boston": 613,
        "New York": 482,
        "Washington": 396,
    },
    "Houston": {
        "Phoenix": 1015,
        "Dallas": 225,
        "Atlanta": 702,
        "Miami": 968,
    },
    "Los Angeles": {
        "San Francisco": 348,
        "Riverside": 50,
        "Phoenix": 357,
    },
    "Miami": {
        "Houston": 968,
        "Washington": 923,
        "Atlanta": 604,
    },
    "New York": {
        "Detroit": 482,
        "Boston": 190,
        "Philadelphia": 81,
    },
    "Philadelphia": {
        "New York": 81,
        "Washington": 123,
    },
    "Phoenix": {
        "Los Angeles": 357,
        "Riverside": 307,
        "Dallas": 887,
        "Houston": 1015,
    },
    "Riverside": {
        "San Francisco": 386,
        "Los Angeles": 50,
        "Phoenix": 307,
        "Chicago": 1704,
    },
    "San Francisco": {
        "Seattle": 678,
        "Los Angeles": 348,
        "Riverside": 386,
    },
    "Seattle": {
        "San Francisco": 678,
        "Chicago": 1737,
    },
    "Washington": {
        "Detroit": 396,
        "Philadelphia": 123,
        "Atlanta": 543,
        "Miami": 923,
    },
}

print(f"Number of cities: {len(graph)}")
print("Cities:", ", ".join(sorted(graph)))

Number of cities: 15
Cities: Atlanta, Boston, Chicago, Dallas, Detroit, Houston, Los Angeles, Miami, New York, Philadelphia, Phoenix, Riverside, San Francisco, Seattle, Washington


In [2]:
# Safety check: the graph should be symmetric because the map is undirected.
# We also verify that all edge costs are positive.
for city, neighbors in graph.items():
    for neighbor, cost in neighbors.items():
        assert neighbor in graph, f"Missing city in graph: {neighbor}"
        assert graph[neighbor].get(city) == cost, f"Asymmetric edge: {city} <-> {neighbor}"
        assert cost > 0, f"Non-positive edge cost: {city} -> {neighbor}"

print("Graph validation passed: undirected and all edge costs are positive.")

Graph validation passed: undirected and all edge costs are positive.


## 3. Build the heuristic safely

We first find the minimum number of edges from every city to the goal using an **unweighted BFS**. If a city is 3 edges away in the fewest-hop sense, then any real route must have at least 3 edges. Since every edge costs at least 50, the remaining cost must be at least 150.

So:

`h(city) = 50 * minimum_hop_count(city, goal)`

This is a lower bound, not an optimistic guess pulled from outside the assignment.

In [3]:
from collections import deque
import heapq
import math

MIN_EDGE_COST = min(
    cost
    for neighbors in graph.values()
    for cost in neighbors.values()
)


def hop_distance_to_goal(graph, goal):
    """Return the minimum number of edges from every reachable city to goal."""
    distances = {goal: 0}
    queue = deque([goal])

    while queue:
        current = queue.popleft()
        for neighbor in graph[current]:
            if neighbor not in distances:
                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

    return distances


def make_heuristic(graph, goal):
    """Create an admissible/consistent graph-only heuristic for a fixed goal."""
    hop_distances = hop_distance_to_goal(graph, goal)

    def heuristic(city):
        if city not in hop_distances:
            return math.inf
        return hop_distances[city] * MIN_EDGE_COST

    return heuristic

print(f"Minimum edge cost on the map = {MIN_EDGE_COST}")

Minimum edge cost on the map = 50


## 4. A* Search implementation

The implementation below uses a **priority queue (`heapq`)**. At every step it chooses the currently most promising city according to `f(n) = g(n) + h(n)`.

A stale-entry check is included because a city can be inserted into the priority queue more than once after a cheaper route is discovered later. This avoids a common implementation bug.

In [4]:
def reconstruct_path(parent, start, goal):
    """Reconstruct the path from start to goal after A* finishes."""
    path = [goal]
    while path[-1] != start:
        path.append(parent[path[-1]])
    path.reverse()
    return path


def a_star(graph, start, goal):
    """Return (shortest_path, shortest_cost, expanded_cities)."""
    if start not in graph:
        raise ValueError(f"Unknown start city: {start}")
    if goal not in graph:
        raise ValueError(f"Unknown goal city: {goal}")

    heuristic = make_heuristic(graph, goal)

    # Best known cost from start to each city.
    g_score = {city: math.inf for city in graph}
    g_score[start] = 0

    # Parent pointers for path reconstruction.
    parent = {}

    # (f_score, g_score, city) gives deterministic tie-breaking.
    open_heap = [(heuristic(start), 0, start)]
    expanded = []

    while open_heap:
        f_score, current_g, current = heapq.heappop(open_heap)

        # Ignore an outdated heap entry.
        if current_g != g_score[current]:
            continue

        expanded.append(current)

        if current == goal:
            path = reconstruct_path(parent, start, goal)
            return path, g_score[goal], expanded

        for neighbor, edge_cost in graph[current].items():
            tentative_g = g_score[current] + edge_cost

            if tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                parent[neighbor] = current
                new_f = tentative_g + heuristic(neighbor)
                heapq.heappush(open_heap, (new_f, tentative_g, neighbor))

    raise ValueError(f"No path exists from {start} to {goal}.")

## 5. Run the assignment example

Because the PDF does not specify which two named cities should be used, this notebook demonstrates the algorithm with:

**Start:** Seattle  
**Goal:** Miami

The code remains general and can solve any pair of cities in the supplied graph.

In [5]:
START_CITY = "Seattle"
GOAL_CITY = "Miami"

path, total_cost, expanded = a_star(graph, START_CITY, GOAL_CITY)

print("Shortest path:")
print(" -> ".join(path))
print(f"Total cost: {total_cost}")
print(f"Cities along path: {len(path)}")
print("Expanded cities:")
print(" -> ".join(expanded))

Shortest path:
Seattle -> Chicago -> Atlanta -> Miami
Total cost: 2929
Cities along path: 4
Expanded cities:
Seattle -> San Francisco -> Los Angeles -> Riverside -> Phoenix -> Chicago -> Detroit -> Dallas -> Atlanta -> Washington -> Houston -> Philadelphia -> New York -> Boston -> Miami


## 6. Verify the exact path cost edge-by-edge

For Seattle -> Miami, the returned route is:

**Seattle -> Chicago -> Atlanta -> Miami**

Its cost is:

`1737 + 588 + 604 = 2929`

In [6]:
def path_cost(graph, path):
    """Calculate the exact total cost of a path."""
    total = 0
    for u, v in zip(path, path[1:]):
        total += graph[u][v]
    return total

print("Edge-by-edge breakdown:")
for u, v in zip(path, path[1:]):
    print(f"{u} -> {v}: {graph[u][v]}")

assert path_cost(graph, path) == total_cost
print()
print(f"Verified total = {path_cost(graph, path)}")

Edge-by-edge breakdown:
Seattle -> Chicago: 1737
Chicago -> Atlanta: 588
Atlanta -> Miami: 604

Verified total = 2929


## 7. Independent optimality check with Dijkstra

Dijkstra's algorithm is used only as an **independent verification**. If A* and Dijkstra return the same minimum cost, we have a strong implementation check that the A* result is truly optimal on this graph.

In [7]:
def dijkstra(graph, start, goal):
    """Independent shortest-path reference implementation."""
    distances = {city: math.inf for city in graph}
    parent = {}
    distances[start] = 0
    heap = [(0, start)]

    while heap:
        current_distance, current = heapq.heappop(heap)

        if current_distance != distances[current]:
            continue

        if current == goal:
            break

        for neighbor, edge_cost in graph[current].items():
            new_distance = current_distance + edge_cost
            if new_distance < distances[neighbor]:
                distances[neighbor] = new_distance
                parent[neighbor] = current
                heapq.heappush(heap, (new_distance, neighbor))

    if distances[goal] == math.inf:
        raise ValueError(f"No path exists from {start} to {goal}.")

    return reconstruct_path(parent, start, goal), distances[goal]

reference_path, reference_cost = dijkstra(graph, START_CITY, GOAL_CITY)

print("Dijkstra path:")
print(" -> ".join(reference_path))
print(f"Dijkstra cost: {reference_cost}")

assert total_cost == reference_cost
assert path_cost(graph, reference_path) == reference_cost
print()
print("Independent verification passed: A* cost equals Dijkstra cost.")

Dijkstra path:
Seattle -> Chicago -> Atlanta -> Miami
Dijkstra cost: 2929

Independent verification passed: A* cost equals Dijkstra cost.


## 8. Final answer to place in the assignment report

### Example shortest path

For the demonstration pair **Seattle to Miami**, A* Search finds:

**Seattle -> Chicago -> Atlanta -> Miami**

The total path cost is **2929**:

- Seattle -> Chicago = 1737
- Chicago -> Atlanta = 588
- Atlanta -> Miami = 604
- **Total = 2929**

The same minimum cost is independently obtained by Dijkstra's algorithm, verifying that the A* result is optimal for this graph.

> **Important:** Because the assignment PDF says "cities A and B" but does not specify A/B on the supplied map, the Seattle -> Miami pair above is explicitly a demonstration. If the lecturer specifies different endpoints, change `START_CITY` and `GOAL_CITY`; the algorithm and graph dictionary do not need to be rewritten.

## 9. Oral assessment - what you should be able to explain

### Question 1: What is A*?
**Answer:** A* is an informed search algorithm that selects nodes using `f(n) = g(n) + h(n)`, where `g(n)` is the cost already travelled and `h(n)` estimates the remaining cost.

### Question 2: Why is this result shortest?
**Answer:** The heuristic used here is admissible because it never overestimates the remaining cost. It is also consistent. Therefore A* can safely return an optimal shortest path. We additionally verified the final cost independently with Dijkstra.

### Question 3: Why is the graph stored in both directions?
**Answer:** The supplied map has undirected edges, so every connection must be represented from both cities.

### Question 4: What does the number on an edge mean?
**Answer:** It is the edge cost. A path cost is the sum of all edge costs along that path.

### Question 5: Why did you not use latitude/longitude as the heuristic?
**Answer:** The assignment supplies edge costs but not city coordinates or a heuristic table. I therefore used a graph-only lower-bound heuristic that is mathematically guaranteed not to overestimate.

## Submission checklist for Part II

- [x] Python dictionary representing the full map
- [x] A* Search algorithm implemented
- [x] Shortest path output
- [x] Total path cost output
- [x] Cities along the path output
- [x] Independent shortest-path verification
- [x] Oral explanation prepared

The assignment's report requirement is to include the Python dictionary, an example shortest path with its cost and cities, and the source code in Jupyter Notebook format.